Instalações bibliotecas

In [3]:
!pip install google-adk -q
!pip install litellm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.4 MB/s eta 0:00:00


Importação das bibliotecas

In [4]:
import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm # For multi-model support
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types # For creating message Content/Parts

import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

print("Libraries imported.")

Libraries imported.


Importação das chaves de API

In [5]:
from dotenv import load_dotenv
import os

load_dotenv() # Colocando as chaves de API em um arquivo .env, para que elas não vazem

print("Google:", bool(os.getenv("GOOGLE_API_KEY")))

print("Groq:", bool(os.getenv("GROQ_API_KEY")))

Google: True
Groq: True


Definindo os modelos que serão utilizados

In [6]:
MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash"

MODEL_GEMINI_2_5_FLASH_LITE = "gemini/gemini-2.5-flash-lite"

MODEL_GROQ = "groq/llama-3.3-70b-versatile"

print("\nEnvironment configured.")


Environment configured.


Definindo a função 'calc_body_metrics'

In [7]:
def calc_body_metrics(metric: str, peso_kg: float, altura_cm: float, idade: int, sexo: str | None = None) -> dict:
    """Calcula IMC ou TMB (Taxa Metabólica Basal) a partir dos dados do usuário.

    Args:
        metric (str): "imc" ou "tmb"
        peso_kg (float): peso em kg
        altura_cm (float): altura em cm
        idade (int): idade em anos
        sexo (str | None): "masculino"/"feminino" (obrigatório para TMB)

    Returns:
        dict: {'status':'success','report':'...'} ou {'status':'error','error_message':'...'}
    """
    print(f"--- Tool: calc_body_metrics called | metric={metric} peso={peso_kg} altura={altura_cm} idade={idade} sexo={sexo} ---")

    # Validações básicas
    if peso_kg <= 0:
        return {"status": "error", "error_message": "Peso inválido. Envie um valor em kg maior que zero."}
    if altura_cm <= 0:
        return {"status": "error", "error_message": "Altura inválida. Envie um valor em cm maior que zero."}
    if idade <= 0:
        return {"status": "error", "error_message": "Idade inválida. Envie um valor em anos maior que zero."}

    metric_norm = (metric or "").strip().lower()
    altura_m = altura_cm / 100.0

    if metric_norm == "imc":
        imc = peso_kg / (altura_m ** 2)

        # Classificação OMS (adultos)
        if imc < 18.5:
            cls = "Abaixo do peso"
        elif imc < 25:
            cls = "Peso normal"
        elif imc < 30:
            cls = "Sobrepeso"
        elif imc < 35:
            cls = "Obesidade grau I"
        elif imc < 40:
            cls = "Obesidade grau II"
        else:
            cls = "Obesidade grau III"

        report = (
            f"IMC calculado: {imc:.2f}\n"
            f"Classificação (OMS): {cls}\n"
            f"Dados usados: peso={peso_kg:.1f} kg, altura={altura_cm:.1f} cm."
        )
        return {"status": "success", "report": report}

    if metric_norm == "tmb":
        if not sexo:
            return {"status": "error", "error_message": "Para calcular TMB, preciso do sexo ('masculino' ou 'feminino')."}

        sexo_norm = sexo.strip().lower()

        # Fórmula de Mifflin–St Jeor
        base = 10 * peso_kg + 6.25 * altura_cm - 5 * idade

        if sexo_norm in ["masculino", "homem", "m", "male"]:
            tmb = base + 5
            sx = "masculino"
        elif sexo_norm in ["feminino", "mulher", "f", "female"]:
            tmb = base - 161
            sx = "feminino"
        else:
            return {"status": "error", "error_message": "Sexo inválido. Use 'masculino' ou 'feminino'."}

        report = (
            f"TMB (Mifflin–St Jeor): {tmb:.0f} kcal/dia\n"
            f"Dados usados: peso={peso_kg:.1f} kg, altura={altura_cm:.1f} cm, idade={idade}, sexo={sx}."
        )
        return {"status": "success", "report": report}

    return {"status": "error", "error_message": "Métrica inválida. Peça 'imc' ou 'tmb'."}


Definindo o Agente (body_agent)

In [8]:
AGENT_MODEL = MODEL_GEMINI_2_5_FLASH  # Começando com o modelo do Gemini para o body_agent

body_agent = Agent(
    name="body_agent_v1",  # Identificador único para o Agente
    model=AGENT_MODEL,  # Especifica qual LLM utilizar
    description="Calcula IMC ou TMB (Taxa Metabólica Basal) com base em dados do usuário.",  # Propósito do agente
    instruction="You are a helpful body metrics assistant. " # Orientações para o agente
                "When the user asks for BMI (IMC), use the 'calc_body_metrics' tool with metric='imc'. "
                "When the user asks for BMR (TMB), use the 'calc_body_metrics' tool with metric='tmb'. "
                "For TMB, if sex is missing, ask the user to provide 'masculino' or 'feminino'. "
                "If the user provides height in meters, convert to centimeters before calling the tool. "
                "If the tool returns an error, inform the user politely. "
                "If the tool is successful, present the report clearly.",
    tools=[calc_body_metrics],  # Ferramentas disponíveis
)

print(f"Agent '{body_agent.name}' created using model '{AGENT_MODEL}'.")


Agent 'body_agent_v1' created using model 'gemini-2.5-flash'.


Configurando o Serviço de Sessão e o Runner

In [9]:
# --- Gerenciamento de Sessão. ---
# Conceito principal: SessionService guarda o histórico e o estado da conversa.
# InMemorySessionService simples, não salva em banco.
session_service = InMemorySessionService()

APP_NAME = "body_metrics_tutorial_app" # Nome da aplicação
USER_ID = "user_1" # Id do usuário, para distinguir sessões de usuários diferentes
SESSION_ID = "session_001" # Identificador fixo de sessão

session = await session_service.create_session( # Cria assíncronamente uma nova sessão
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# --- Runner ---
# Conceito principal: o Runner controla o ciclo de execução do agente.
runner = Runner(
    agent=body_agent, # O agente que irá ser rodado
    app_name=APP_NAME,   # Associa as execuções do Runner à aplicação
    session_service=session_service # Usa o gerenciamento de sessão
)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created: App='body_metrics_tutorial_app', User='user_1', Session='session_001'
Runner created for agent 'body_agent_v1'.


Definindo a função de Interação do Agente

In [10]:
from google.genai import types

async def call_agent_async(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")


  content = types.Content(role='user', parts=[types.Part(text=query)]) # Prepara a mensagem do usuário no formado que o ADK exige

  final_response_text = "Agent did not produce a final response." # Default

  # run_async executa a lógica do agente e produz eventos
  # Percorre os eventos até encontrar a resposta final
  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content): # Loop que executa o agente e recebe eventos gerados durante a execução
      print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}") # Permite ver todos os eventos durante a execução

      # is_final_response() indica o evento que encerra a resposta do agente
      if event.is_final_response(): # Verifica se o evento atual é a resposta final do agente
          if event.content and event.content.parts: # Garante que o evento contém conteúdo e que esse conteúdo possui partes
             # Assumindo que o texto da reposta está na primeira parte, extrai ele e armazena como reposta final
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate: # Caso não haja texto, verifica erros
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          break # Interrompe o loop assim que a resposta final é encontrada

  print(f"<<< Agent Response: {final_response_text}")

Rodando a conversação inicial

In [11]:
# É necessário uma função assíncrona para usar await
async def run_conversation():  # Executa várias interações com o agente
    await call_agent_async("Tenho 80 kg, 1,75 m e 25 anos. Calcule meu IMC.",
                           runner=runner,
                           user_id=USER_ID,
                           session_id=SESSION_ID)

    await call_agent_async("Agora calcule minha TMB. Tenho 80 kg, 175 cm, 25 anos.",
                           runner=runner,
                           user_id=USER_ID,
                           session_id=SESSION_ID)  # Espera-se um erro pedindo sexo

    await call_agent_async("Sou masculino. Pode calcular a TMB com os mesmos dados (80 kg, 175 cm, 25 anos)?",
                           runner=runner,
                           user_id=USER_ID,
                           session_id=SESSION_ID)

# Executa a conversa usando await em um contexto assíncrono
await run_conversation()



>>> User Query: Tenho 80 kg, 1,75 m e 25 anos. Calcule meu IMC.


  [Event] Author: body_agent_v1, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'altura_cm': 175,
      'idade': 25,
      'metric': 'imc',
      'peso_kg': 80
    },
    id='adk-ad16ca31-4ac3-416c-9809-adfcb0520ebf',
    name='calc_body_metrics'
  ),
  thought_signature=b"\n\xb6\x03\x01\xbe>\xf6\xfb\xc3\xd7J:\r9+\xc7\\-f\xac\n\xce/\x85'^[\xe1\xb0\xc4\xf8\xe2>\x06\x0fL_\xefJAa\x1a\x8b\xfc\xad\x90\xa3\xa4\xc7\xae\xf9\x11\xa7(?\xa0\xb1}/\x90\xb8h:\x15/'T\xc7\xd8#\xc1\x13*:\x87\x0e0\x1a\xa3mT\xa7\x16#\xf2\xeb\xde\xdb\xae0\xd2\x88\x89P\xfd\x99\xb8...'
)] role='model'
--- Tool: calc_body_metrics called | metric=imc peso=80 altura=175 idade=25 sexo=None ---
  [Event] Author: body_agent_v1, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='adk-ad16ca31-4ac3-416c-9809-adfcb0520ebf',
    name='calc_body_metrics',
    response={
      'report': """IMC calculado: 26.12
Classificação (OMS): Sobrepeso
Dado

Importando o LiteLlm

In [12]:
from google.adk.models.lite_llm import LiteLlm

Definindo e testando o Agente Gemini

In [15]:
# --- Agente usando o gemini-2.5-flash-lite ---
body_agent_gemini = None # Inicializa a variável do agente como None
runner_gemini = None      # Inicializa a variável do runner como None

try:
    body_agent_gemini = Agent(
        name="body_agent_gemini", # Define um nome para o Agente
        model=LiteLlm(model=MODEL_GEMINI_2_5_FLASH_LITE), # Configura o agente para usar um modelo via LiteLLM
        description="Calcula IMC/TMB (using gemini-2.5-flash-lite).", # Descrição curta do que o agente faz
        instruction="You are a helpful body metrics assistant powered by gemini-2.5-flash-lite. " # Instrução para orientar o agente
                    "Use the 'calc_body_metrics' tool for city weather requests. "
                    "Clearly present successful reports or polite error messages based on the tool's output status.",
        tools=[calc_body_metrics], # Usa novamente a mesma ferramenta
    )
    print(f"Agent '{body_agent_gemini.name}' created using model '{MODEL_GEMINI_2_5_FLASH_LITE}'.")

    # InMemorySessionService simples, não salva em banco.
    session_service_gemini = InMemorySessionService() # Cria um servidor dedicado

    APP_NAME_GEMINI = "weather_tutorial_app_gemini" # Nome da aplicação
    USER_ID_GEMINI = "user_1_gemini" # Id do usuário, para distinguir sessões de usuários diferentes
    SESSION_ID_GEMINI = "session_001_gemini" # Identificador fixo de sessão

    # Cria a sessão específica a conversa irá ocorrer
    session_gemini = await session_service_gemini.create_session(
        app_name=APP_NAME_GEMINI,
        user_id=USER_ID_GEMINI,
        session_id=SESSION_ID_GEMINI
    )
    print(f"Session created: App='{APP_NAME_GEMINI}', User='{USER_ID_GEMINI}', Session='{SESSION_ID_GEMINI}'")

    # Agora cria o Runner específico desse agente e desse serviço de sessão
    runner_gemini = Runner( # Instancia o Runner que vai executar o agente em loop de eventos.
        agent=body_agent_gemini,
        app_name=APP_NAME_GEMINI,       # Usa o nome da aplicação
        session_service=session_service_gemini # Usa a sessão específica
        )
    print(f"Runner created for agent '{runner_gemini.agent.name}'.")

    # --- Testa o gemini Agente ---
    print("\n--- Testing gemini Agent ---")
    await call_agent_async(query = "Sou homem, tenho 20 anos, 1,81 metros de altura e 78kg. Calcule meu IMC.",
                           runner=runner_gemini,
                           user_id=USER_ID_GEMINI,
                           session_id=SESSION_ID_GEMINI)

except Exception as e:
    print(f"❌ Could not create or run gemini agent '{MODEL_GEMINI_2_5_FLASH_LITE}'. Check API Key and model name. Error: {e}")

Agent 'body_agent_gemini' created using model 'gemini/gemini-2.5-flash-lite'.
Session created: App='weather_tutorial_app_gemini', User='user_1_gemini', Session='session_001_gemini'
Runner created for agent 'body_agent_gemini'.

--- Testing gemini Agent ---

>>> User Query: Sou homem, tenho 20 anos, 1,81 metros de altura e 78kg. Calcule meu IMC.
  [Event] Author: body_agent_gemini, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'altura_cm': 181,
      'idade': 20,
      'metric': 'imc',
      'peso_kg': 78,
      'sexo': {
        'masculino': ''
      }
    },
    id='call_a154706a13b74958b89454cca712',
    name='calc_body_metrics'
  )
)] role='model'
--- Tool: calc_body_metrics called | metric=imc peso=78 altura=181 idade=20 sexo={'masculino': ''} ---
  [Event] Author: body_agent_gemini, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='call_a154706a13b74958b89454cca712',
    name='calc_body_

Definindo e Testando o Agente Groq

In [16]:
# --- Agent usando o Groq ---
body_agent_groq = None # Inicializa a variável do agente como None
runner_groq = None        # Inicializa a variável do runner como None

try:
    body_agent_groq = Agent(
        name="body_agent_groq", # Define um nome para o Agente
        model=LiteLlm(model=MODEL_GROQ), # Configura o agente para usar um modelo via LiteLLM
        description="Calcula IMC/TMB (using Groq).", # Descrição curta do que o agente faz
        instruction="You are a helpful body metrics assistant powered by Groq. " # Instrução para orientar o agente
                    "Use the 'calc_body_metrics' tool for city weather requests. "
                    "Analyze the tool's dictionary output ('status', 'report'/'error_message'). "
                    "Clearly present successful reports or polite error messages.",
        tools=[calc_body_metrics], # Usa novamente a mesma ferramenta
    )
    print(f"Agent '{body_agent_groq.name}' created using model '{MODEL_GROQ}'.")

    # InMemorySessionService simples, não salva em banco.
    session_service_groq = InMemorySessionService() # Cria um servidor dedicado

    APP_NAME_GROQ = "weather_tutorial_app_groq" # Nome da aplicação
    USER_ID_GROQ = "user_1_groq" # Id do usuário, para distinguir sessões de usuários diferentes
    SESSION_ID_GROQ = "session_001_groq" # Identificador fixo de sessão

    # Cria a sessão específica a conversa irá ocorrer
    session_groq = await session_service_groq.create_session(
        app_name=APP_NAME_GROQ,
        user_id=USER_ID_GROQ,
        session_id=SESSION_ID_GROQ
    )
    print(f"Session created: App='{APP_NAME_GROQ}', User='{USER_ID_GROQ}', Session='{SESSION_ID_GROQ}'")

    # Agora cria o Runner específico desse agente e desse serviço de sessão
    runner_groq = Runner( # Instancia o Runner que vai executar o agente em loop de eventos.
        agent=body_agent_groq,
        app_name=APP_NAME_GROQ,       # Usa o nome da aplicação
        session_service=session_service_groq # Usa a sessão específica
        )
    print(f"Runner created for agent '{runner_groq.agent.name}'.")

    # --- Testa o Agente Groq ---
    print("\n--- Testing Groq Agent ---")
    await call_agent_async(query = "Sou homem, tenho 20 anos, 1,81 metros de altura e 78kg. Calcule meu TMB.",
                           runner=runner_groq,
                           user_id=USER_ID_GROQ,
                           session_id=SESSION_ID_GROQ)

except Exception as e:
    print(f"❌ Could not create or run Claude agent '{MODEL_GROQ}'. Check API Key and model name. Error: {e}")

Agent 'body_agent_groq' created using model 'groq/llama-3.3-70b-versatile'.
Session created: App='weather_tutorial_app_groq', User='user_1_groq', Session='session_001_groq'
Runner created for agent 'body_agent_groq'.

--- Testing Groq Agent ---

>>> User Query: Sou homem, tenho 20 anos, 1,81 metros de altura e 78kg. Calcule meu TMB.
  [Event] Author: body_agent_groq, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'altura_cm': 181,
      'idade': 20,
      'metric': 'tmb',
      'peso_kg': 78,
      'sexo': 'masculino'
    },
    id='tnssb1myw',
    name='calc_body_metrics'
  )
)] role='model'
--- Tool: calc_body_metrics called | metric=tmb peso=78 altura=181 idade=20 sexo=masculino ---
  [Event] Author: body_agent_groq, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='tnssb1myw',
    name='calc_body_metrics',
    response={
      'report': """TMB (Mifflin–St Jeor): 1816 kcal/dia
Dados usados:

Definindo Ferramentas para os Agentes de Boas-vindas e despedidas

In [17]:
from typing import Optional

def say_hello(name: Optional[str] = None) -> str:
    """Provides a simple greeting. If a name is provided, it will be used.

    Args:
        name (str, optional): The name of the person to greet. Defaults to a generic greeting if not provided.

    Returns:
        str: A friendly greeting message.
    """
    if name: # Verifica se name possui um valor considerado verdadeiro
        greeting = f"Hello, {name}!"
        print(f"--- Tool: say_hello called with name: {name} ---")
    else: # Executado quando name não foi fornecido ou é None
        greeting = "Hello there!"
        print(f"--- Tool: say_hello called without a specific name (name_arg_value: {name}) ---")
    return greeting

def say_goodbye() -> str: # Uma simples função para retornar uma despedida
    """Provides a simple farewell message to conclude the conversation."""
    print(f"--- Tool: say_goodbye called ---")
    return "Goodbye! Have a great day."

print("Greeting and Farewell tools defined.")

# Teste
print(say_hello("Alice"))
print(say_hello()) # Teste sem argumento
print(say_hello(name=None)) # Teste com 'name' sendo definido como None explicitamente

Greeting and Farewell tools defined.
--- Tool: say_hello called with name: Alice ---
Hello, Alice!
--- Tool: say_hello called without a specific name (name_arg_value: None) ---
Hello there!
--- Tool: say_hello called without a specific name (name_arg_value: None) ---
Hello there!


Definindo os sub-agentes de boas-vindas e de despedida

In [18]:
# --- Agente de Boas-vindas ---
greeting_agent = None # Inicializa a variável do agente como None
try:
    greeting_agent = Agent(
        model = MODEL_GEMINI_2_5_FLASH,
        name="greeting_agent", # Define um nome para o Agente
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting to the user. " # Instrução para orientar o agente
                    "Use the 'say_hello' tool to generate the greeting. "
                    "If the user provides their name, make sure to pass it to the tool. "
                    "Do not engage in any other conversation or tasks.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.", # Descrição curta do que o agente faz, mas crucial para a delegação
        tools=[say_hello], # Lista das ferramentas disponíveis para uso
    )
    print(f"✅ Agent '{greeting_agent.name}' created using model '{greeting_agent.model}'.")
except Exception as e:
    print(f"❌ Could not create Greeting agent. Check API Key ({greeting_agent.model}). Error: {e}")

# --- Agente de despedida ---
farewell_agent = None # Inicializa a variável do agente como None
try:
    farewell_agent = Agent(
        model = MODEL_GEMINI_2_5_FLASH,
        name="farewell_agent", # Define um nome para o Agente
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message. " # Instrução para orientar o agente
                    "Use the 'say_goodbye' tool when the user indicates they are leaving or ending the conversation "
                    "(e.g., using words like 'bye', 'goodbye', 'thanks bye', 'see you'). "
                    "Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.", # Descrição curta do que o agente faz, mas crucial para a delegação
        tools=[say_goodbye], # Lista das ferramentas disponíveis para uso
    )
    print(f"✅ Agent '{farewell_agent.name}' created using model '{farewell_agent.model}'.")
except Exception as e:
    print(f"❌ Could not create Farewell agent. Check API Key ({farewell_agent.model}). Error: {e}")

✅ Agent 'greeting_agent' created using model 'gemini-2.5-flash'.
✅ Agent 'farewell_agent' created using model 'gemini-2.5-flash'.


Definindo o Agente Raiz com os sub-agentes

In [19]:
root_agent = None  # Inicializa a variável do agente raiz como None
runner_root = None  # Inicializa a variável do runner do agente raiz como None

if greeting_agent and farewell_agent and 'calc_body_metrics' in globals():  # Verifica se sub-agentes e tool existem
    root_agent_model = MODEL_GEMINI_2_5_FLASH

    body_agent_team = Agent(
        name="body_agent_v2",
        model=root_agent_model,
        description="The main coordinator agent. Handles IMC/TMB requests and delegates greetings/farewells to specialists.",
        instruction="You are the main Body Metrics Agent coordinating a team. "
                    "Your primary responsibility is to calculate IMC (BMI) or TMB (BMR) from user data. "
                    "Use the 'calc_body_metrics' tool ONLY when the user is asking for IMC/TMB and provides (or you can infer) peso, altura and idade. "
                    "If the user asks for TMB and sex is missing, ask for it ('masculino' or 'feminino'). "
                    "If the user provides height in meters, convert to centimeters before calling the tool. "
                    "You have specialized sub-agents: "
                    "1. 'greeting_agent': Handles simple greetings like 'Hi', 'Hello'. Delegate to it for these. "
                    "2. 'farewell_agent': Handles simple farewells like 'Bye', 'See you'. Delegate to it for these. "
                    "Analyze the user's query. If it's a greeting, delegate to 'greeting_agent'. "
                    "If it's a farewell, delegate to 'farewell_agent'. "
                    "If it's an IMC/TMB request, handle it yourself using 'calc_body_metrics'. "
                    "For anything else, respond appropriately or state you cannot handle it.",
        tools=[calc_body_metrics],
        sub_agents=[greeting_agent, farewell_agent]
    )
    print(f"✅ Root Agent '{body_agent_team.name}' created using model '{root_agent_model}' with sub-agents: {[sa.name for sa in body_agent_team.sub_agents]}")
else:
    print("❌ Cannot create root team agent. Prerequisites missing.")
    if not greeting_agent: print(" - greeting_agent definition missing.")
    if not farewell_agent: print(" - farewell_agent definition missing.")
    if 'calc_body_metrics' not in globals(): print(" - calc_body_metrics tool missing.")


✅ Root Agent 'body_agent_v2' created using model 'gemini-2.5-flash' with sub-agents: ['greeting_agent', 'farewell_agent']


Interagindo com o time de agentes

In [20]:
import asyncio

root_agent_var_name = 'root_agent' # Define o nome padrão da variável
if 'body_agent_team' in globals(): # Verifica se existe uma variável global chamada body_agent_team
    root_agent_var_name = 'body_agent_team' # Se existir, atualiza o nome da variável “fonte” do root agent para body_agent_team
elif 'root_agent' not in globals(): # Se body_agent_team não existe e também não existe root_agent, então o agente raiz não está disponível
    print("⚠️ Root agent ('root_agent' or 'body_agent_team') not found. Cannot define run_team_conversation.")
    root_agent = None # Define root_agent como None para evitar NameError caso alguma parte referencie root_agent depois.

if root_agent_var_name in globals() and globals()[root_agent_var_name]: # Checa se o nome da variável está no escopo global e se essa variável não é None
    async def run_team_conversation():
        print("\n--- Testing Agent Team Delegation ---")
        session_service = InMemorySessionService() # Cria um serviço de sessão em memória para armazenar o histórico
        APP_NAME = "body_metrics_tutorial_agent_team" # Define o nome da aplicação
        USER_ID = "user_1_agent_team" # Define o ID do usuário específico desse teste
        SESSION_ID = "session_001_agent_team" # Define o ID da sessão específica desse teste
        session = await session_service.create_session( # Cria a sessão onde a conversa acontecerá
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )
        print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

        actual_root_agent = globals()[root_agent_var_name] # Busca o objeto do root agent diretamente do dicionário global
        runner_agent_team = Runner( # Cria um Runner que vai executar o root agent
            agent=actual_root_agent,
            app_name=APP_NAME,
            session_service=session_service
        )
        print(f"Runner created for agent '{actual_root_agent.name}'.")

        # Interações
        await call_agent_async(query = "Hello there!",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)
        await call_agent_async(query = "Tenho 90 kg, 180 cm e 30 anos. Calcule meu IMC.",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)
        await call_agent_async(query = "Thanks, bye!",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)

    # Execução
    print("Attempting execution using 'await' (default for notebooks)...")
    await run_team_conversation()

else: # Caso o root agent não exista/esteja None
    print("\n⚠️ Skipping agent team conversation execution as the root agent was not successfully defined in a previous step.")

Attempting execution using 'await' (default for notebooks)...

--- Testing Agent Team Delegation ---
Session created: App='body_metrics_tutorial_agent_team', User='user_1_agent_team', Session='session_001_agent_team'
Runner created for agent 'body_agent_v2'.

>>> User Query: Hello there!
  [Event] Author: body_agent_v2, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'agent_name': 'greeting_agent'
    },
    id='adk-5c9cb420-fbc3-4e1e-bc6a-2faa5ed01463',
    name='transfer_to_agent'
  ),
  thought_signature=b'\n\x89\x02\x01\xbe>\xf6\xfb\x94\x00G\xfa\xc8{l\r!W\x11\xd4\xe3\xe7\x86\xeb\xcaJ\x0e\xceV\x05t\x1b\xbf\xdb\xd69\x8dk\x0c;!\x11f\xc3\xc5X\x07\xe3\xef\x8bS\xa0\xd2\xbf\xbc\xb3{\xebM\xa6\xdeIC\xb4c\xf5h9\x8f"\xb5\xc6\x01\'W\x9d\x81\x17\x86(\x87\xd8\xd9)\x7fe\x15\xe3(\xa0\x01\x1f\n\x80w!\xec...'
)] role='model'
  [Event] Author: body_agent_v2, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='a

_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 29.213969128s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_

Iniciando um novo serviço de sessão e estado

In [21]:
from google.adk.sessions import InMemorySessionService

# Criando uma nova instância de serviço de sessão para demonstrar estado
session_service_stateful = InMemorySessionService() # Cria uma nova instância do serviço de sessão
print("✅ New InMemorySessionService created for state demonstration.")

# Define um novo ID de sessão e de usuário para essa parte
SESSION_ID_STATEFUL = "session_state_demo_001"
USER_ID_STATEFUL = "user_state_demo"

# Define os dados do estado inicial - O usuário prefere IMC como padrão
initial_state = {
    "user_preference_default_metric": "imc"
}

# Cria a sessão, indicando o estado inicial
session_stateful = await session_service_stateful.create_session(
    app_name=APP_NAME, # Associa a sessão ao mesmo nome de aplicação usado anteriormente
    user_id=USER_ID_STATEFUL, # Associa a sessão ao usuário específico dessa demonstração
    session_id=SESSION_ID_STATEFUL, # Define o identificador único da sessão
    state=initial_state # Inicializa o estado durante a criação
)
print(f"✅ Session '{SESSION_ID_STATEFUL}' created for user '{USER_ID_STATEFUL}'.")

# Verifica se o estado inicial está configurado corretamente
retrieved_session = await session_service_stateful.get_session(app_name=APP_NAME,
                                                         user_id=USER_ID_STATEFUL,
                                                         session_id = SESSION_ID_STATEFUL)
print("\n--- Initial Session State ---")
if retrieved_session:
    print(retrieved_session.state)
else:
    print("Error: Could not retrieve session.")

✅ New InMemorySessionService created for state demonstration.
✅ Session 'session_state_demo_001' created for user 'user_state_demo'.

--- Initial Session State ---
{'user_preference_default_metric': 'imc'}


In [22]:
from google.adk.tools.tool_context import ToolContext

def calc_body_metrics_stateful(metric: str, peso_kg: float, altura_cm: float, idade: int, tool_context: ToolContext, sexo: str | None = None) -> dict:
    """Calcula IMC/TMB e salva informações úteis no estado da sessão."""
    print(f"--- Tool: calc_body_metrics_stateful called | metric={metric} peso={peso_kg} altura={altura_cm} idade={idade} sexo={sexo} ---")

    # Salva no estado qual foi a última métrica pedida
    tool_context.state["last_metric_requested"] = metric
    print(f"--- Tool: Updated state 'last_metric_requested': {metric} ---")

    # Reusa a ferramenta stateless para manter mudanças mínimas
    result = calc_body_metrics(metric=metric, peso_kg=peso_kg, altura_cm=altura_cm, idade=idade, sexo=sexo)

    # Salva o último resultado (texto) no estado
    if result.get("status") == "success":
        tool_context.state["last_body_report_by_tool"] = result.get("report")
    else:
        tool_context.state["last_body_report_by_tool"] = result.get("error_message")
    print("--- Tool: Updated state 'last_body_report_by_tool' ---")

    return result

print("✅ State-aware 'calc_body_metrics_stateful' tool defined.")


✅ State-aware 'calc_body_metrics_stateful' tool defined.


Redefinindo os sub-agentes e atualizando o agente raiz com output_key

In [23]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner

# Redefinindo o agente de boas vindas
greeting_agent = None
try:
    greeting_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="greeting_agent",
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.",
        tools=[say_hello],
    )
    print(f"✅ Agent '{greeting_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Greeting agent. Error: {e}")

# Redefinindo o agente de depesdida
farewell_agent = None
try:
    farewell_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="farewell_agent",
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye' tool. Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
        tools=[say_goodbye],
    )
    print(f"✅ Agent '{farewell_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Farewell agent. Error: {e}")

# Definindo o Agente raiz atualizado
root_agent_stateful = None # Inicializa a variável do agente raiz como None
runner_root_stateful = None # Inicializa a variável do runner do agente raiz como None

# Verifica os pré-requisitos antes da criação do agente raiz
if greeting_agent and farewell_agent and 'calc_body_metrics_stateful' in globals():

    root_agent_model = MODEL_GEMINI_2_5_FLASH # Modelo que será o orquestrador

    root_agent_stateful = Agent(
        name="body_agent_v4_stateful", # Define o nome do agente raiz
        model=root_agent_model,
        description="Main agent: Provides IMC/TMB (stateful), delegates greetings/farewells, saves report to state.", # Descreve o papel do agente raiz, incluindo uso de estado e delegação.
        instruction="You are the main Body Metrics Agent. Your job is to calculate IMC/TMB using 'calc_body_metrics_stateful'. " # Instruções que definem o comportamento principal do agente raiz
                    "The tool may store helpful info in session state (e.g., last metric requested). "
                    "Delegate simple greetings to 'greeting_agent' and farewells to 'farewell_agent'. "
                    "Handle only IMC/TMB requests, greetings, and farewells.",
        tools=[calc_body_metrics_stateful], # Registra a tool de IMC/TMB que lê e escreve no estado da sessão
        sub_agents=[greeting_agent, farewell_agent], # Inclue os sub-agentes
        output_key="last_body_report" # Define a chave do estado onde a resposta final de clima será salva automaticamente
    )
    print(f"✅ Root Agent '{root_agent_stateful.name}' created using stateful tool and output_key.")

    # Criação do runner
    runner_root_stateful = Runner( # Cria o Runner responsável por executar o agente raiz com estado
        agent=root_agent_stateful,
        app_name=APP_NAME,
        session_service=session_service_stateful # Usa o serviço de sessão com estado (onde preferências e respostas são armazenadas)
    )
    print(f"✅ Runner created for stateful root agent '{runner_root_stateful.agent.name}' using stateful session service.")

else:
    print("❌ Cannot create stateful root agent. Prerequisites missing.")
    if not greeting_agent: print(" - greeting_agent definition missing.")
    if not farewell_agent: print(" - farewell_agent definition missing.")
    if 'calc_body_metrics_stateful' not in globals(): print(" - calc_body_metrics_stateful tool missing.")

✅ Agent 'greeting_agent' redefined.
✅ Agent 'farewell_agent' redefined.
✅ Root Agent 'body_agent_v4_stateful' created using stateful tool and output_key.
✅ Runner created for stateful root agent 'body_agent_v4_stateful' using stateful session service.


Testando o fluxo da conversa e o output_key

In [25]:
import asyncio

if 'runner_root_stateful' in globals() and runner_root_stateful: # Verifica se a variável global runner_root_stateful existe e se ela não é None
    async def run_stateful_conversation(): # Declara a função assíncrona que vai executar os “turnos” da conversa e testar o estado
        print("\n--- Testing State: State Writes & output_key (IMC/TMB) ---")

        # 1. Verificar IMC
        print("--- Turn 1: Requesting IMC (expect success) ---")
        await call_agent_async(query= "Tenho 70 kg, 170 cm e 22 anos. Calcule o IMC.", # Envia a pergunta ao agente
                               runner=runner_root_stateful,
                               user_id=USER_ID_STATEFUL,
                               session_id=SESSION_ID_STATEFUL
                              )

        # 2. A preferência será alterada manualmente para TMB (mexendo diretamente no storage em memória)
        print("\n--- Manually Updating State: Setting default metric to 'tmb' ---")
        try:
            # Acessar sessions[...] é algo específico do InMemorySessionService e serve só para testes
            stored_session = session_service_stateful.sessions[APP_NAME][USER_ID_STATEFUL][SESSION_ID_STATEFUL] # Acessa diretamente a sessão guardada dentro do dicionário interno do serviço em memória
            stored_session.state["user_preference_default_metric"] = "tmb" # Atualiza a preferência no state para Fahrenheit
            print(f"--- Stored session state updated. Current 'user_preference_default_metric': {stored_session.state.get('user_preference_default_metric', 'Not Set')} ---") # Added .get for safety
        except KeyError:
            print(f"--- Error: Could not retrieve session '{SESSION_ID_STATEFUL}' from internal storage for user '{USER_ID_STATEFUL}' in app '{APP_NAME}' to update state. Check IDs and if session was created. ---")
        except Exception as e:
             print(f"--- Error updating internal session state: {e} ---")

        # 3. Verifica a preferência (Deve ser usado TMB agora)
        # Isso também irá atualizar 'last_body_report' via output_key
        print("\n--- Turn 2: Requesting TMB (expect ask for sex or error if missing) ---")
        await call_agent_async(query= "Agora calcule minha TMB. Tenho 70 kg, 170 cm e 22 anos.",
                               runner=runner_root_stateful,
                               user_id=USER_ID_STATEFUL,
                               session_id=SESSION_ID_STATEFUL
                              )

        # 4. Testa delegação
        # Isso irá atualizar 'last_body_report' novamente
        print("\n--- Turn 3: Sending a greeting ---")
        await call_agent_async(query= "Hi!", # Envia uma saudação; o root agent deve delegar ao greeting_agent
                               runner=runner_root_stateful,
                               user_id=USER_ID_STATEFUL,
                               session_id=SESSION_ID_STATEFUL
                              )

    print("Attempting execution using 'await' (default for notebooks)...")
    await run_stateful_conversation() # Executa toda a sequência de testes


    # Inspeciona o estado da sessão final depois da conversa
    # Este bloco é executado após a conclusão de qualquer um dos métodos de execução
    print("\n--- Inspecting Final Session State ---")
    final_session = await session_service_stateful.get_session(app_name=APP_NAME, # Recupera a sessão do serviço de sessão
                                                         user_id= USER_ID_STATEFUL,
                                                         session_id=SESSION_ID_STATEFUL)
    if final_session: # Verifica se a sessão foi recuperada com sucesso
        # Usa .get() para evitar erro se alguma chave não existir
        print(f"Final Preference: {final_session.state.get('user_preference_default_metric', 'Not Set')}")
        print(f"Final Last Body Report (from output_key): {final_session.state.get('last_body_report', 'Not Set')}")
        print(f"Final Last Metric Requested (by tool): {final_session.state.get('last_metric_requested', 'Not Set')}")
    else:
        print("\n❌ Error: Could not retrieve final session state.")

else:
    print("\n⚠️ Skipping state test conversation. Stateful root agent runner ('runner_root_stateful') is not available.")

Attempting execution using 'await' (default for notebooks)...

--- Testing State: State Writes & output_key (IMC/TMB) ---
--- Turn 1: Requesting IMC (expect success) ---

>>> User Query: Tenho 70 kg, 170 cm e 22 anos. Calcule o IMC.
  [Event] Author: greeting_agent, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'agent_name': 'body_agent_v4_stateful'
    },
    id='adk-ca7e995c-8f07-4d4e-831d-9439e3e3069a',
    name='transfer_to_agent'
  ),
  thought_signature=b'\n\x83\x04\x01\xbe>\xf6\xfb\xb9\xfa\x7f;\xe0\xc6&T\xeaj\xb3\x04\xc6\x87\xb9\x9f\x18"%\x17\xdaP\x1cf(\x13b?\x1e1\xbd\x95 \xef\xa5\xad^D\xa0\xc9\xe0\xb5\xfc\x1f\x15]]\x0f\x8d\xaf\x84o \x0b\t)y\x0b\x87\x17\x8e\xec\x1a\x94\xd4u\x9d\x1c\x9a\x80\x13\xc2\xe3\xdcx\xe8\x81L\xc3=\x90\x11\x82\xa5\xaaM^\xca\x80...'
)] role='model'
  [Event] Author: greeting_agent, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='adk-ca7e995c-8f07-4d4e-831d-9439e3

_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 4.418567199s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_

Define o before_model_callback que funciona como barreira de segurança

In [26]:
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types
from typing import Optional

def block_keyword_guardrail( # Define a função de callback que atuará como guardrail
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]: # Define que a função pode retornar um LlmResponse (bloqueando a chamada) ou None (permitindo a continuação)

    agent_name = callback_context.agent_name # Obtém o nome do agente cujo chamado ao modelo está sendo interceptado
    print(f"--- Callback: block_keyword_guardrail running for agent: {agent_name} ---")

    # Extrai o texto da última mensagem do usuário no histórico de requisição
    last_user_message_text = ""
    if llm_request.contents: # Verifica se a requisição contém histórico de mensagens
        for content in reversed(llm_request.contents): # Itera sobre as mensagens da requisição de trás para frente
            if content.role == 'user' and content.parts: # Verifica se a mensagem é do usuário e se possui partes de conteúdo
                if content.parts[0].text: # Confirma que a primeira parte contém texto
                    last_user_message_text = content.parts[0].text # Armazena o texto da última mensagem do usuário
                    break # Interrompe o loop após encontrar a mensagem de usuário mais recente

    print(f"--- Callback: Inspecting last user message: '{last_user_message_text[:100]}...' ---")

    # Lógica Barreira de proteção
    keyword_to_block = "BLOCK" # Define a palavra-chave que deve acionar o bloqueio
    if keyword_to_block in last_user_message_text.upper(): # Verifica se a palavra “BLOCK” aparece na mensagem
        print(f"--- Callback: Found '{keyword_to_block}'. Blocking LLM call! ---")
        callback_context.state["guardrail_block_keyword_triggered"] = True # Escreve no estado da sessão que o guardrail foi acionado
        print(f"--- Callback: Set state 'guardrail_block_keyword_triggered': True ---")

        # Construindo e retornando um LlmResponse para interromper o fluxo
        return LlmResponse(
            content=types.Content(
                role="model", # Mimic a response from the agent's perspective
                parts=[types.Part(text=f"I cannot process this request because it contains the blocked keyword '{keyword_to_block}'.")], # Cria o texto da resposta informando que a requisição foi bloqueada pela palavra-chave
            )

        )
    else: # Executado quando a palavra bloqueada não é encontrada
        print(f"--- Callback: Keyword not found. Allowing LLM call for {agent_name}. ---")
        return None # sinaliza ao ADK que o processamento deve continuar normalmente

print("✅ block_keyword_guardrail function defined.")

✅ block_keyword_guardrail function defined.


Atualizando o Agenre raiz com o before_model_callback

In [27]:
# Redefinindo os sub-agentes
greeting_agent = None
try:
    greeting_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="greeting_agent", # Mantendo o nome original para consistência
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.",
        tools=[say_hello],
    )
    print(f"✅ Sub-Agent '{greeting_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Greeting agent. Check Model/API Key ({greeting_agent.model}). Error: {e}")

farewell_agent = None
try:
    farewell_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="farewell_agent", # Mantendo o nome original
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye' tool. Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
        tools=[say_goodbye],
    )
    print(f"✅ Sub-Agent '{farewell_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Farewell agent. Check Model/API Key ({farewell_agent.model}). Error: {e}")


# Definindo o agente raiz com o callback
root_agent_model_guardrail = None
runner_root_model_guardrail = None

if greeting_agent and farewell_agent and 'calc_body_metrics_stateful' in globals() and 'block_keyword_guardrail' in globals(): # Só continua se os sub-agentes existem, a tool stateful existe e o callback guardrail existe

    root_agent_model = MODEL_GEMINI_2_5_FLASH

    root_agent_model_guardrail = Agent(
        name="body_agent_v5_model_guardrail", # Nova versão do nome para clareza
        model=root_agent_model,
        description="Main agent: Handles IMC/TMB, delegates greetings/farewells, includes input keyword guardrail.",
        instruction="You are the main Weather Agent. Provide weather using 'calc_body_metrics_stateful'. "
                    "Delegate simple greetings to 'greeting_agent' and farewells to 'farewell_agent'. "
                    "Handle only IMC/TMB requests, greetings, and farewells.",
        tools=[calc_body_metrics_stateful], # Registra a tool de clima com estado
        sub_agents=[greeting_agent, farewell_agent], # Conecta os sub-agentes redefinidos ao root agent para delegação
        output_key="last_body_report", # Configura a chave do state onde a resposta final do turno será salva automaticamente
        before_model_callback=block_keyword_guardrail # Registra o callback que roda antes de chamar o modelo
    )
    print(f"✅ Root Agent '{root_agent_model_guardrail.name}' created with before_model_callback.")

    # Criando um Runner usando o mesmo serviço de sessão com estado
    if 'session_service_stateful' in globals(): # Verifica se o serviço de sessão com estado existe no escopo global
        runner_root_model_guardrail = Runner(
            agent=root_agent_model_guardrail,
            app_name=APP_NAME, # Associa o Runner ao nome da aplicação
            session_service=session_service_stateful # Usa o serviço de sessão com estado
        )
        print(f"✅ Runner created for guardrail agent '{runner_root_model_guardrail.agent.name}', using stateful session service.")
    else:
        print("❌ Cannot create runner. 'session_service_stateful' from Step 4 is missing.")

else: # Executado se algum pré-requisito (sub-agentes, tool, callback) estiver faltando
    print("❌ Cannot create root agent with model guardrail. One or more prerequisites are missing or failed initialization:")
    if not greeting_agent: print("   - Greeting Agent")
    if not farewell_agent: print("   - Farewell Agent")
    if 'calc_body_metrics_stateful' not in globals(): print("   - 'calc_body_metrics_stateful' tool")
    if 'block_keyword_guardrail' not in globals(): print("   - 'block_keyword_guardrail' callback")

✅ Sub-Agent 'greeting_agent' redefined.
✅ Sub-Agent 'farewell_agent' redefined.
✅ Root Agent 'body_agent_v5_model_guardrail' created with before_model_callback.
✅ Runner created for guardrail agent 'body_agent_v5_model_guardrail', using stateful session service.


Testando a entrada do modelo com Guardrail

In [28]:
import asyncio

if 'runner_root_model_guardrail' in globals() and runner_root_model_guardrail:
    async def run_guardrail_test_conversation():
        print("\n--- Testing Model Input Guardrail (keyword) ---")

        interaction_func = lambda query: call_agent_async(
            query,
            runner_root_model_guardrail,
            USER_ID_STATEFUL,
            SESSION_ID_STATEFUL
        )

        # 1. Interação normal (deve passar no guardrail)
        print("--- Turn 1: Requesting IMC (expect allowed) ---")
        await interaction_func("Tenho 65 kg, 165 cm e 21 anos. Calcule meu IMC.")

        # 2. Interação com palavra bloqueada (callback deve interceptar)
        print("\n--- Turn 2: Requesting with blocked keyword (expect blocked) ---")
        await interaction_func("BLOCK esta solicitação de TMB para mim (70 kg, 170 cm, 22 anos).")

        # 3. Saudação (deve passar e delegar)
        print("\n--- Turn 3: Sending a greeting (expect allowed) ---")
        await interaction_func("Hello again")

    print("Attempting execution using 'await' (default for notebooks)...")
    await run_guardrail_test_conversation()

    # Inspecionando o estado final da sessão após o teste
    print("\n--- Inspecting Final Session State (After Guardrail Test) ---")
    final_session = await session_service_stateful.get_session(
        app_name=APP_NAME,
        user_id=USER_ID_STATEFUL,
        session_id=SESSION_ID_STATEFUL
    )
    if final_session:
        print(f"Guardrail Triggered Flag: {final_session.state.get('guardrail_block_keyword_triggered', 'Not Set (or False)')}")
        print(f"Last Body Report (from output_key): {final_session.state.get('last_body_report', 'Not Set')}")
        print(f"Default Metric Preference: {final_session.state.get('user_preference_default_metric', 'Not Set')}")
    else:
        print("\n❌ Error: Could not retrieve final session state.")

else:
    print("\n⚠️ Skipping model guardrail test. Runner ('runner_root_model_guardrail') is not available.")


Attempting execution using 'await' (default for notebooks)...

--- Testing Model Input Guardrail (keyword) ---
--- Turn 1: Requesting IMC (expect allowed) ---

>>> User Query: Tenho 65 kg, 165 cm e 21 anos. Calcule meu IMC.


_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 8.239310463s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_

Definindo a barrieira de proteção before_tool_callback

In [29]:
from google.adk.tools.base_tool import BaseTool
from google.adk.tools.tool_context import ToolContext
from typing import Optional, Dict, Any

def block_unrealistic_args_tool_guardrail(
    tool: BaseTool, args: Dict[str, Any], tool_context: ToolContext
) -> Optional[Dict]:
    """Bloqueia chamadas à tool com argumentos irreais/fora do razoável (demonstração de guardrail)."""
    print(f"--- Callback: block_unrealistic_args_tool_guardrail running for tool: {tool.name} with args: {args} ---")

    target_tool_name = "calc_body_metrics_stateful"
    if tool.name != target_tool_name:
        return None

    # Extraindo args com segurança
    peso = args.get("peso_kg")
    altura = args.get("altura_cm")
    idade = args.get("idade")

    # Regras simples de plausibilidade (apenas demonstração)
    # (não é validação médica, apenas evita inputs absurdos)
    if peso is not None and (peso < 20 or peso > 400):
        msg = "Entrada bloqueada: 'peso_kg' fora do intervalo plausível (20–400 kg)."
        print(f"--- Callback: Blocking tool call. Reason: {msg} ---")
        tool_context.state["guardrail_block_unrealistic_args_triggered"] = True
        return {"status": "error", "error_message": msg}

    if altura is not None and (altura < 100 or altura > 250):
        msg = "Entrada bloqueada: 'altura_cm' fora do intervalo plausível (100–250 cm)."
        print(f"--- Callback: Blocking tool call. Reason: {msg} ---")
        tool_context.state["guardrail_block_unrealistic_args_triggered"] = True
        return {"status": "error", "error_message": msg}

    if idade is not None and (idade < 5 or idade > 120):
        msg = "Entrada bloqueada: 'idade' fora do intervalo plausível (5–120 anos)."
        print(f"--- Callback: Blocking tool call. Reason: {msg} ---")
        tool_context.state["guardrail_block_unrealistic_args_triggered"] = True
        return {"status": "error", "error_message": msg}

    print("--- Callback: Args look plausible. Allowing tool call. ---")
    return None

print("✅ block_unrealistic_args_tool_guardrail function defined.")


✅ block_unrealistic_args_tool_guardrail function defined.


Atualizando o agente raiz com ambos os callbacks

In [30]:
# Redefinindo os sub-agentes
greeting_agent = None
try:
    greeting_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="greeting_agent", # Mantém o nome original para consistência
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.",
        tools=[say_hello],
    )
    print(f"✅ Sub-Agent '{greeting_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Greeting agent. Check Model/API Key ({greeting_agent.model}). Error: {e}")

farewell_agent = None
try:
    farewell_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="farewell_agent", # Mantém o nome original
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye' tool. Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
        tools=[say_goodbye],
    )
    print(f"✅ Sub-Agent '{farewell_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Farewell agent. Check Model/API Key ({farewell_agent.model}). Error: {e}")

# Definindo o Agente raiz com ambos os callbacks
root_agent_tool_guardrail = None
runner_root_tool_guardrail = None

if ('greeting_agent' in globals() and greeting_agent and # Verifica se a variável greeting_agent existe globalmente e se ela contém um objeto válido
    'farewell_agent' in globals() and farewell_agent and # Verifica o mesmo para o farewell_agent
    'calc_body_metrics_stateful' in globals() and # Verifica se a tool calc_body_metrics_stateful foi definida
    'block_keyword_guardrail' in globals() and # Verifica se o callback de guardrail antes do modelo existe
    'block_unrealistic_args_tool_guardrail' in globals()): # Verifica se o callback de guardrail antes da tool existe

    root_agent_model = MODEL_GEMINI_2_5_FLASH

    root_agent_tool_guardrail = Agent(
        name="body_agent_v6_tool_guardrail", # Nova versão do nome
        model=root_agent_model,
        description="Main agent: Handles IMC/TMB, delegates, includes input AND tool guardrails.",
        instruction="You are the main Body Metrics Agent. Provide IMC/TMB using 'calc_body_metrics_stateful'. "
                    "Delegate greetings to 'greeting_agent' and farewells to 'farewell_agent'. "
                    "Handle only IMC/TMB, greetings, and farewells.",
        tools=[calc_body_metrics_stateful],
        sub_agents=[greeting_agent, farewell_agent],
        output_key="last_body_report",
        before_model_callback=block_keyword_guardrail, # Mantém o guardrail de modelo
        before_tool_callback=block_unrealistic_args_tool_guardrail # Adiciona o guardrail de ferrameta
    )
    print(f"✅ Root Agent '{root_agent_tool_guardrail.name}' created with BOTH callbacks.")

    # Criando o Runner reaproveitando o mesmo serviço de sessão stateful
    if 'session_service_stateful' in globals(): # Verifica se o serviço de sessão com estado existe
        runner_root_tool_guardrail = Runner(
            agent=root_agent_tool_guardrail,
            app_name=APP_NAME,
            session_service=session_service_stateful # Usa o mesmo serviço de sessão, preservando o estado e histórico
        )
        print(f"✅ Runner created for tool guardrail agent '{runner_root_tool_guardrail.agent.name}', using stateful session service.")
    else:
        print("❌ Cannot create runner. 'session_service_stateful' from Step 4/5 is missing.")

else:
    print("❌ Cannot create root agent with tool guardrail. Prerequisites missing.")

✅ Sub-Agent 'greeting_agent' redefined.
✅ Sub-Agent 'farewell_agent' redefined.
✅ Root Agent 'body_agent_v6_tool_guardrail' created with BOTH callbacks.
✅ Runner created for tool guardrail agent 'body_agent_v6_tool_guardrail', using stateful session service.


Testando a barreira de proteção do argumento da ferramenta

In [31]:
import asyncio

if 'runner_root_tool_guardrail' in globals() and runner_root_tool_guardrail:
    async def run_tool_guardrail_test():
        print("\n--- Testing Tool Argument Guardrail (unrealistic args blocked) ---")

        interaction_func = lambda query: call_agent_async(
            query,
            runner_root_tool_guardrail,
            USER_ID_STATEFUL,
            SESSION_ID_STATEFUL
        )

        # 1. Args plausíveis (deve passar)
        print("--- Turn 1: Requesting IMC with plausible values (expect allowed) ---")
        await interaction_func("Tenho 80 kg, 180 cm e 28 anos. Calcule meu IMC.")

        # 2. Args absurdos: altura muito grande (deve ser bloqueado pelo before_tool_callback)
        print("\n--- Turn 2: Requesting IMC with unrealistic values (expect blocked by tool guardrail) ---")
        await interaction_func("Tenho 80 kg, 999 cm e 28 anos. Calcule meu IMC.")

        # 3. Volta a funcionar com args plausíveis
        print("\n--- Turn 3: Requesting TMB (expect allowed; may ask for sex) ---")
        await interaction_func("Agora calcule minha TMB com 80 kg, 180 cm e 28 anos. Sou masculino.")

    print("Attempting execution using 'await' (default for notebooks)...")
    await run_tool_guardrail_test()

    # Inspecionando o estado final da sessão após o teste
    print("\n--- Inspecting Final Session State (After Tool Guardrail Test) ---")
    final_session = await session_service_stateful.get_session(
        app_name=APP_NAME,
        user_id=USER_ID_STATEFUL,
        session_id=SESSION_ID_STATEFUL
    )
    if final_session:
        print(f"Tool Guardrail Triggered Flag: {final_session.state.get('guardrail_block_unrealistic_args_triggered', 'Not Set (or False)')}")
        print(f"Last Body Report (from output_key): {final_session.state.get('last_body_report', 'Not Set')}")
        print(f"Last Body Report (by tool): {final_session.state.get('last_body_report_by_tool', 'Not Set')}")
        print(f"Last Metric Requested (by tool): {final_session.state.get('last_metric_requested', 'Not Set')}")
    else:
        print("\n❌ Error: Could not retrieve final session state.")

else:
    print("\n⚠️ Skipping tool guardrail test. Runner ('runner_root_tool_guardrail') is not available.")


Attempting execution using 'await' (default for notebooks)...

--- Testing Tool Argument Guardrail (unrealistic args blocked) ---
--- Turn 1: Requesting IMC with plausible values (expect allowed) ---

>>> User Query: Tenho 80 kg, 180 cm e 28 anos. Calcule meu IMC.


_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 3.370268443s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_